# PLS Regression - Custom Voltage Feature Selection

**Mục tiêu:** Cho phép tự chọn:
- Khoảng điện thế `V_MIN` → `V_MAX`
- Bước nhảy `V_STEP`

Sau đó tự động build dataset, chạy PLS với LOO-CV và hiển thị kết quả đầy đủ.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict, LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries loaded')

---
## ⚙️ CẤU HÌNH - Chỉnh tại đây!

In [ ]:
# ================================================================
#  ⚙️  THAY ĐỔI CÁC THÔNG SỐ NÀY
# ================================================================

FILE_PATH = 'CUCOMOF.csv'   # Đường dẫn tới file CSV gốc

# --- Khoảng điện thế muốn dùng làm feature ---
V_MIN  = 0.60   # V (ví dụ: -0.2, 0.0, 0.5, 0.6 ...)
V_MAX  = 0.80   # V (ví dụ:  0.8, 0.7, 0.75 ...)
V_STEP = 0.02   # Bước nhảy (ví dụ: 0.1, 0.05, 0.02, 0.01)

# --- Số PLS components muốn thử (None = tự động chọn tốt nhất) ---
N_COMPONENTS = None   # None hoặc 1, 2, 3 ...

# ================================================================
print(f'Config: V = [{V_MIN}, {V_MAX}], step = {V_STEP}')
n_expected = len(np.arange(V_MIN, V_MAX + V_STEP/2, V_STEP))
print(f'Số features dự kiến: {n_expected}')

---
## 1. Load & Parse Raw Data

In [ ]:
# Nồng độ glucose (mM) - label của từng column
CONCENTRATIONS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
CONC_COLS = [f'{c}mM' for c in CONCENTRATIONS]

# Parse file thủ công (xử lý tab kép trong raw data)
data_rows = []
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line or i in [0, 1, 2]:   # bỏ 3 dòng header
            continue
        parts = [p.strip() for p in line.split('\t')]
        clean = [p for p in parts if p != '']
        if len(clean) >= 11:
            clean = clean[:11]
        else:
            clean += [np.nan] * (11 - len(clean))
        data_rows.append(clean)

df_raw = pd.DataFrame(data_rows, columns=['V'] + CONC_COLS)
df_raw = df_raw.apply(pd.to_numeric, errors='coerce')

# Chỉ lấy chiều quét đi (-0.2V → 0.8V)
v_max_idx = df_raw['V'].idxmax()
df_fwd = df_raw.iloc[:v_max_idx+1].copy().reset_index(drop=True)

print(f'Tổng số điểm đo (forward sweep): {len(df_fwd)}')
print(f'V range: {df_fwd["V"].min():.4f}V → {df_fwd["V"].max():.4f}V')
print(f'Nồng độ glucose: {CONCENTRATIONS} mM')

---
## 2. Build Feature Matrix theo config

In [ ]:
# Tạo danh sách V targets
target_V = np.arange(V_MIN, V_MAX + V_STEP/2, V_STEP)
target_V = np.round(target_V, 6)

# Kiểm tra range hợp lệ
v_data_min = df_fwd['V'].min()
v_data_max = df_fwd['V'].max()
assert V_MIN >= v_data_min - 0.01, f'V_MIN={V_MIN} nằm ngoài data ({v_data_min:.3f}V)'
assert V_MAX <= v_data_max + 0.01, f'V_MAX={V_MAX} nằm ngoài data ({v_data_max:.3f}V)'
assert V_STEP > 0, 'V_STEP phải > 0'

# Với mỗi target V, lấy hàng gần nhất trong data
selected_rows = []
matched_V = []
for tv in target_V:
    idx = (df_fwd['V'] - tv).abs().idxmin()
    row = df_fwd.iloc[idx]
    selected_rows.append(row[CONC_COLS].values)
    matched_V.append(row['V'])

# X: shape (n_samples=10, n_features=n_V_points)
# Mỗi hàng = 1 nồng độ glucose; mỗi cột = I tại 1 V
feature_labels = [f'V_{tv:.3f}' for tv in target_V]
X = np.array(selected_rows).T          # (10 mẫu, n features)
y = np.array(CONCENTRATIONS)           # (10,)

df_features = pd.DataFrame(X, columns=feature_labels)
df_features.insert(0, 'glucose_mM', y)

print(f'✅ Feature matrix built:')
print(f'   Số mẫu    : {X.shape[0]}')
print(f'   Số features: {X.shape[1]}')
print(f'   V range    : {target_V[0]:.3f}V → {target_V[-1]:.3f}V  (step={V_STEP}V)')
print(f'   V thực tế  : {[round(v,3) for v in matched_V]}')
print()
df_features.round(3)

---
## 3. Xem CV Curves tại vùng đã chọn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.cm.plasma
line_colors = cmap(np.linspace(0.1, 0.9, len(CONCENTRATIONS)))

# Toàn bộ CV
ax = axes[0]
for i, (col, conc) in enumerate(zip(CONC_COLS, CONCENTRATIONS)):
    ax.plot(df_fwd['V'], df_fwd[col], color=line_colors[i], label=f'{conc}mM', linewidth=1.4)
ax.axvspan(V_MIN, V_MAX, alpha=0.12, color='red', label=f'Selected [{V_MIN},{V_MAX}]V')
for tv in target_V:
    ax.axvline(x=tv, color='red', linewidth=0.5, alpha=0.4)
ax.set_xlabel('Potential (V)', fontsize=11)
ax.set_ylabel('Current (µA)', fontsize=11)
ax.set_title('Full CV - Forward Sweep', fontsize=12, fontweight='bold')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

# Zoom vào vùng đã chọn
ax2 = axes[1]
margin = (V_MAX - V_MIN) * 0.3
mask = (df_fwd['V'] >= V_MIN - margin) & (df_fwd['V'] <= V_MAX + margin)
df_zoom = df_fwd[mask]
for i, (col, conc) in enumerate(zip(CONC_COLS, CONCENTRATIONS)):
    ax2.plot(df_zoom['V'], df_zoom[col], color=line_colors[i], label=f'{conc}mM', linewidth=1.8)
for tv, mv in zip(target_V, matched_V):
    ax2.axvline(x=mv, color='red', linewidth=1.2, alpha=0.6, linestyle='--')
ax2.axvspan(V_MIN, V_MAX, alpha=0.08, color='red')
ax2.set_xlabel('Potential (V)', fontsize=11)
ax2.set_ylabel('Current (µA)', fontsize=11)
ax2.set_title(f'Zoom: [{V_MIN}V, {V_MAX}V]  step={V_STEP}V  ({X.shape[1]} features)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=7, ncol=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cv_curves_selected_V.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Univariate Analysis: R² & Sensitivity tại từng V

In [ ]:
from sklearn.linear_model import LinearRegression

r2_per_V, sens_per_V, pearson_per_V = [], [], []
for j, label in enumerate(feature_labels):
    I_j = X[:, j]
    if np.any(np.isnan(I_j)):
        r2_per_V.append(np.nan); sens_per_V.append(np.nan); pearson_per_V.append(np.nan)
        continue
    lr = LinearRegression().fit(y.reshape(-1,1), I_j)
    r2_per_V.append(r2_score(I_j, lr.predict(y.reshape(-1,1))))
    sens_per_V.append(lr.coef_[0])
    pearson_per_V.append(pearsonr(I_j, y)[0])

analysis_df = pd.DataFrame({
    'V_target': target_V,
    'V_actual': [round(v,4) for v in matched_V],
    'R2_linear': np.round(r2_per_V, 4),
    'Pearson_r': np.round(pearson_per_V, 4),
    'Sensitivity_uA_per_mM': np.round(sens_per_V, 4)
})

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
v_labels = [f'{tv:.3f}' for tv in target_V]

for ax, col, color, title in zip(
    axes,
    ['R2_linear', 'Pearson_r', 'Sensitivity_uA_per_mM'],
    ['steelblue', 'purple', 'green'],
    ['R² (linear fit)', 'Pearson r', 'Sensitivity (µA/mM)']
):
    ax.bar(v_labels, analysis_df[col], color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
    ax.set_xlabel('Voltage (V)', fontsize=10)
    ax.set_ylabel(title, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=60)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Per-Voltage Univariate Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('univariate_per_V.png', dpi=150, bbox_inches='tight')
plt.show()

print(analysis_df.sort_values('R2_linear', ascending=False).to_string(index=False))

---
## 5. Chọn số PLS components tối ưu (LOO-CV)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

max_comp = min(X.shape[1], len(y) - 1)
loo = LeaveOneOut()

comp_results = []
for nc in range(1, max_comp + 1):
    pls = PLSRegression(n_components=nc)
    y_pred = cross_val_predict(pls, X_scaled, y, cv=loo).ravel()
    r2  = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    comp_results.append({'n_comp': nc, 'R2_LOO': round(r2,4), 'RMSE_LOO': round(rmse,4)})

comp_df = pd.DataFrame(comp_results)

# Chọn n_components
if N_COMPONENTS is None:
    best_nc = comp_df.loc[comp_df['R2_LOO'].idxmax(), 'n_comp']
    print(f'Auto-selected n_components = {best_nc}')
else:
    best_nc = N_COMPONENTS
    print(f'Manual n_components = {best_nc}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(comp_df['n_comp'], comp_df['R2_LOO'], 'bo-', linewidth=2, markersize=8)
ax1.axvline(x=best_nc, color='red', linestyle='--', label=f'Best={best_nc}')
for _, row in comp_df.iterrows():
    ax1.annotate(f"{row['R2_LOO']:.3f}", (row['n_comp'], row['R2_LOO']),
                 textcoords='offset points', xytext=(4, 6), fontsize=8)
ax1.set_xlabel('n_components'); ax1.set_ylabel('R² (LOO-CV)')
ax1.set_title('R² vs Components'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(comp_df['n_comp'], comp_df['RMSE_LOO'], 'rs-', linewidth=2, markersize=8)
ax2.axvline(x=best_nc, color='red', linestyle='--', label=f'Best={best_nc}')
for _, row in comp_df.iterrows():
    ax2.annotate(f"{row['RMSE_LOO']:.3f}", (row['n_comp'], row['RMSE_LOO']),
                 textcoords='offset points', xytext=(4, 6), fontsize=8)
ax2.set_xlabel('n_components'); ax2.set_ylabel('RMSE (mM)')
ax2.set_title('RMSE vs Components'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pls_components.png', dpi=150, bbox_inches='tight')
plt.show()
print(comp_df.to_string(index=False))

---
## 6. Train PLS & Đánh giá

In [ ]:
pls = PLSRegression(n_components=best_nc)
pls.fit(X_scaled, y)

y_train = pls.predict(X_scaled).ravel()
y_loo   = cross_val_predict(pls, X_scaled, y, cv=LeaveOneOut()).ravel()

r2_train  = r2_score(y, y_train)
r2_loo    = r2_score(y, y_loo)
rmse_train = np.sqrt(mean_squared_error(y, y_train))
rmse_loo   = np.sqrt(mean_squared_error(y, y_loo))

print('=' * 45)
print(f'  PLS Model  (n_components={best_nc})')
print('=' * 45)
print(f'  Features : {X.shape[1]} V points  [{V_MIN}→{V_MAX}V, step={V_STEP}]')
print(f'  Train  R² = {r2_train:.4f}   RMSE = {rmse_train:.4f} mM')
print(f'  LOO-CV R² = {r2_loo:.4f}   RMSE = {rmse_loo:.4f} mM')
print('=' * 45)

---
## 7. Visualization kết quả

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, hspace=0.42, wspace=0.35)

# --- 7.1 Predicted vs Actual (LOO) ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(y, y_loo, s=100, color='steelblue', zorder=5)
lim = [y.min()-0.4, y.max()+0.4]
ax1.plot(lim, lim, 'r--', linewidth=1.5, label='Ideal')
for a, p in zip(y, y_loo):
    ax1.annotate(f'{a}mM', (a, p), textcoords='offset points', xytext=(5,4), fontsize=8)
ax1.set_xlabel('Actual (mM)'); ax1.set_ylabel('Predicted (mM)')
ax1.set_title(f'Predicted vs Actual (LOO)\nR²={r2_loo:.4f}, RMSE={rmse_loo:.4f}mM', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

# --- 7.2 Residuals ---
ax2 = fig.add_subplot(gs[0, 1])
residuals = y_loo - y
bar_colors = ['#e74c3c' if r < 0 else '#3498db' for r in residuals]
ax2.bar(y, residuals, width=0.08, color=bar_colors, alpha=0.8, edgecolor='black')
ax2.axhline(0, color='black', linewidth=1)
ax2.set_xlabel('Actual (mM)'); ax2.set_ylabel('Residual (mM)')
ax2.set_title('Residuals (LOO-CV)', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# --- 7.3 VIP Scores ---
ax3 = fig.add_subplot(gs[0, 2])
def vip_scores(model):
    t = model.x_scores_; w = model.x_weights_; q = model.y_loadings_
    p, h = w.shape
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)
    vips = np.array([np.sqrt(p * (s.T @ np.array([(w[i,j]/np.linalg.norm(w[:,j]))**2
                     for j in range(h)])) / total_s)[0] for i in range(p)])
    return vips

vip = vip_scores(pls)
vip_colors = ['#e74c3c' if v >= 1.0 else '#95a5a6' for v in vip]
short_labels = [f'{tv:.2f}' for tv in target_V]
ax3.bar(short_labels, vip, color=vip_colors, alpha=0.85, edgecolor='black', linewidth=0.5)
ax3.axhline(1.0, color='red', linestyle='--', linewidth=1.5, label='VIP=1.0')
ax3.set_xlabel('Voltage (V)'); ax3.set_ylabel('VIP Score')
ax3.set_title('VIP Scores', fontweight='bold')
ax3.tick_params(axis='x', rotation=60)
ax3.legend(); ax3.grid(True, alpha=0.3, axis='y')

# --- 7.4 PLS Loadings (W*) ---
ax4 = fig.add_subplot(gs[1, 0])
for nc_i in range(best_nc):
    ax4.plot(short_labels, pls.x_rotations_[:, nc_i], 'o-', linewidth=1.8,
             markersize=6, label=f'Component {nc_i+1}')
ax4.axhline(0, color='black', linewidth=0.8)
ax4.set_xlabel('Voltage (V)'); ax4.set_ylabel('Loading weight')
ax4.set_title('PLS X-Loadings (W*)', fontweight='bold')
ax4.tick_params(axis='x', rotation=60)
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)

# --- 7.5 I vs [Glucose] tại V có VIP cao nhất ---
ax5 = fig.add_subplot(gs[1, 1])
best_vip_idx = np.argmax(vip)
I_best = X[:, best_vip_idx]
from sklearn.linear_model import LinearRegression
lr = LinearRegression().fit(y.reshape(-1,1), I_best)
r2_uni = r2_score(I_best, lr.predict(y.reshape(-1,1)))
ax5.scatter(y, I_best, s=80, color='darkorange', zorder=5)
y_fit = lr.predict(np.linspace(y.min(), y.max(), 100).reshape(-1,1))
ax5.plot(np.linspace(y.min(), y.max(), 100), y_fit, 'r-', linewidth=1.5)
ax5.set_xlabel('[Glucose] (mM)'); ax5.set_ylabel('Current (µA)')
ax5.set_title(f'I vs [Glucose] @ V={target_V[best_vip_idx]:.3f}V (top VIP)\nR²={r2_uni:.4f}', fontweight='bold')
ax5.grid(True, alpha=0.3)

# --- 7.6 Scores Plot T1 vs T2 (nếu có ≥2 comp) ---
ax6 = fig.add_subplot(gs[1, 2])
if best_nc >= 2:
    T = pls.x_scores_
    sc = ax6.scatter(T[:,0], T[:,1], c=y, cmap='viridis', s=120, edgecolor='black', zorder=5)
    plt.colorbar(sc, ax=ax6, label='Glucose (mM)')
    for i, conc in enumerate(y):
        ax6.annotate(f'{conc}mM', (T[i,0], T[i,1]), textcoords='offset points', xytext=(6,4), fontsize=8)
    ax6.set_xlabel('Component 1'); ax6.set_ylabel('Component 2')
    ax6.set_title('PLS Scores Plot (T1 vs T2)', fontweight='bold')
    ax6.grid(True, alpha=0.3)
else:
    T = pls.x_scores_
    sc = ax6.scatter(y, T[:,0], c=y, cmap='viridis', s=120, edgecolor='black', zorder=5)
    for i, conc in enumerate(y):
        ax6.annotate(f'{conc}mM', (conc, T[i,0]), textcoords='offset points', xytext=(6,4), fontsize=8)
    ax6.set_xlabel('[Glucose] (mM)'); ax6.set_ylabel('Component 1')
    ax6.set_title('PLS Score T1 vs [Glucose]', fontweight='bold')
    ax6.grid(True, alpha=0.3)

plt.suptitle(
    f'PLS Results  |  V=[{V_MIN},{V_MAX}]  step={V_STEP}  ({X.shape[1]} features)  '
    f'nComp={best_nc}  |  R²(LOO)={r2_loo:.4f}  RMSE={rmse_loo:.4f}mM',
    fontsize=11, fontweight='bold'
)
plt.savefig('pls_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Bảng kết quả & Export

In [ ]:
results = pd.DataFrame({
    'Actual (mM)'       : y,
    'Pred_Train (mM)'   : y_train.round(4),
    'Pred_LOO (mM)'     : y_loo.round(4),
    'Residual_LOO'      : (y_loo - y).round(4),
    'AbsError_LOO'      : np.abs(y_loo - y).round(4)
})

print('=== Prediction Results ===')
print(results.to_string(index=False))
print(f'\nMean Abs Error (LOO): {results["AbsError_LOO"].mean():.4f} mM')
print(f'R²  (LOO-CV)        : {r2_loo:.4f}')
print(f'RMSE (LOO-CV)       : {rmse_loo:.4f} mM')

# Export feature matrix
export_name = f'features_V{V_MIN}to{V_MAX}_step{V_STEP}.csv'
df_features.to_csv(export_name, index=False)
print(f'\n✅ Feature matrix saved: {export_name}')